#### 지도학습 기반 감성 분석 실습 - IMDB 영화평

In [16]:
import pandas as pd

review_df = pd.read_csv('./labeledTrainData.tsv', header=0, sep='\t', quoting=3)
review_df.head(3)

,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."


In [17]:
print(review_df['review'][0])

"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.<br /><br />Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.<br /><br />The actual feature film bit when it finally sta

**데이터 사전 처리 html태그 제거 및 숫자문자 제거**

In [18]:
import re

# <br> html 태그는 replace 함수로 공백으로 변환
review_df['review'] = review_df['review'].str.replace('<br />', ' ') # 판다스의 Series는 replace를 함수를 쓰지 못하므로 str 으로 변경 후 사용


# 파이썬의 정규 표현식 모듈인 re 를 이용하여 영어 문자열이 아닌 문자는 모두 공백으로 변환
review_df['review'] = review_df['review'].apply( lambda x : re.sub("[^a-zA-Z]", " ", x) )

**학습/테스트 데이터 분리**

In [19]:
from sklearn.model_selection import train_test_split

class_df = review_df['sentiment']
feature_df = review_df.drop(['id', 'sentiment'], axis=1, inplace=False)

X_train, X_test, y_train, y_test = train_test_split(feature_df, class_df, test_size=0.3, random_state=56)

X_train.shape, X_test.shape

((17500, 1), (7500, 1))

**Pipleline을 통해 Count기반 피처 벡터화 및 머신러닝 학습/예측/평가**

In [20]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

pipeline = Pipeline([
    ('cnt_vect', CountVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('lr_clf', LogisticRegression(C=10))
])

pipeline.fit(X_train['review'], y_train)
pred = pipeline.predict(X_test['review'])
pred_probs = pipeline.predict_proba(X_test['review'])[:, 1]

print('예측 정확도는 {0:.4f}, ROU-AUC는 {1:.4f}'.format(accuracy_score(y_test, pred), roc_auc_score(y_test, pred_probs)))



/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


예측 정확도는 0.8857, ROU-AUC는 0.9498


**Pipleline을 통해 TF-IDF 기반 피처 벡터화 및 머신러닝 학습/예측/평가**

In [21]:
pipeline = Pipeline([
    ('tf_idf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('lr_clf', LogisticRegression(C=10))
])

pipeline.fit(X_train['review'], y_train)

pred = pipeline.predict(X_test['review'])
pred_probs = pipeline.predict_proba(X_test['review'])[:, 1]

print('예측 정확도는 {0:.4f}, ROU-AUC는 {1:.4f}'.format(accuracy_score(y_test, pred), roc_auc_score(y_test, pred_probs)))



/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


예측 정확도는 0.8961, ROU-AUC는 0.9585


#### SentiWordNet 을 이용한 감성 분석
1. 문서(Document)를 문장(Sentence) 단위로 분해
2. 다시 문장을 단어(Word) 단위로 토큰화하고 품사 태깅 POS
3. 품사 태깅된 단어 기반으로 synset 객체와 senti_synset 객체를 생성
4. Senti_synset 에서 긍정 감성/부정 감성 지수를 구하고 이를 모두 합산해 특정 임계치 값 이상일 때 긍정 감성으로, 그렇지 않을 때는 부정 감성으로 결정

VADER lexicon을 이용한 Sentiment Analysis

In [27]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')


senti_analyzer = SentimentIntensityAnalyzer()
senti_scores = senti_analyzer.polarity_scores(review_df['review'][0])
print(senti_scores)

{'neg': 0.13, 'neu': 0.743, 'pos': 0.127, 'compound': -0.7943}


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/songbeom/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [33]:
def vader_polarity(review, threshold=0.1):
    analyer = SentimentIntensityAnalyzer()
    score = analyer.polarity_scores(review)
    
    # compound 값에 기반하여 threshold 이상이면 1(긍정), 이하 -1(부정), 그 외는 0(중립) 반환
    agg_score = score['compound']
    final_sentiment = 1 if agg_score >= threshold else 0
    return final_sentiment

review_df['vader_sentiment'] = review_df['review'].apply(lambda x: vader_polarity(x, threshold=0.1))
y_target = review_df['sentiment'].values
y_vader_pred = review_df['vader_sentiment'].values

In [34]:
print('### VADER 예측 성능 평가 ###')
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

print(confusion_matrix(y_target, y_vader_pred))
print('정확도:', accuracy_score(y_target, y_vader_pred))
print('정밀도:', precision_score(y_target, y_vader_pred))
print('재현율:', recall_score(y_target, y_vader_pred))
print('F1 점수:', f1_score(y_target, y_vader_pred))
print('ROC AUC 점수:', roc_auc_score(y_target, y_vader_pred))

### VADER 예측 성능 평가 ###
[[ 6747  5753]
 [ 1858 10642]]
정확도: 0.69556
정밀도: 0.6491003354681305
재현율: 0.85136
F1 점수: 0.7365980273403703
ROC AUC 점수: 0.6955600000000001
